In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import spearmanr
import nbformat

from dazzle_dash.clean_dataset import nettoyer
from dazzle_dash.ratio_fixe_graph import courbe_avec_outliers


original = pd.read_csv('data/life_expectancy_data.csv')
df = nettoyer(original)

### **Les outils d'aide à la création de graphiques**

Ici nous avons créé un fichier de détection des zéros. En effet nous devons être en mesure de décider si les zéros doivent être transformés en NaN (par exemple pour ne pas influencer le calcul de la médiane, éléments central du calcul du ratio plus loin), ou rester à zéro (comme élement relevé dans un pays : par exemple zéro cas de rougeole).

In [ ]:
import numpy as np
import pandas as pd


def diagnostic_zeros(df, colonne, mini_valeurs=3):
    """Diagnostic des zéros et des séries figées pour UNE colonne.

    Affiche un bilan global, puis le détail par pays.
    mini_valeurs : nombre minimum de valeurs renseignées pour tester
                   si une série est figée (constante).
    """
    serie = df[colonne]

    # --- Bilan global ---
    nb_zeros = (serie == 0).sum()
    print(f"--- {colonne} ---")
    print(f"Zéros : {nb_zeros} ({100 * nb_zeros / len(serie):.1f} %)")
    print(f"NaN   : {serie.isna().sum()}")

    if nb_zeros == 0:
        print("Aucun zéro à convertir.\n")
        return None

    # --- Détail par pays ---
    lignes = []
    for pays, bloc in df.groupby("Country"):
        bloc = bloc.sort_values("Year")        # ordre chronologique indispensable
        valeurs = bloc[colonne]
        est_zero = (valeurs == 0).values       # tableau de True/False

        if not est_zero.any():
            continue                           # ce pays n'a aucun zéro, on passe

        # nb_renseignees : valeurs non NaN (contrairement à len(), qui compte les NaN)
        nb_renseignees = valeurs.notna().sum()

        # Les zéros sont-ils tous collés au début de la série ?
        # est_zero[:n].all() = les n premières valeurs sont toutes des zéros
        nb_zeros_pays = est_zero.sum()
        debut_serie = est_zero[:nb_zeros_pays].all()

        # Série figée : toutes les valeurs renseignées sont identiques
        # (on ignore les zéros pour ce test, sinon ils fausseraient le compte)
        sans_zero = valeurs.replace(0, np.nan)
        figee = (sans_zero.notna().sum() >= mini_valeurs
                 and sans_zero.nunique() == 1)

        lignes.append({
            "pays": pays,
            "nb_zeros": nb_zeros_pays,
            "nb_renseignees": nb_renseignees,
            "tout_a_zero": nb_zeros_pays == nb_renseignees,
            "debut_serie": debut_serie,
            "figee_hors_zeros": figee,
        })

    detail = pd.DataFrame(lignes).set_index("pays")

    # --- Synthèse : c'est elle qui oriente la décision ---
    print(f"Pays touchés        : {len(detail)} / {df['Country'].nunique()}")
    print(f"  dont tout à zéro  : {detail['tout_a_zero'].sum()}")
    print(f"  dont zéros en début de série : {detail['debut_serie'].sum()}")
    print(f"  dont série figée hors zéros  : {detail['figee_hors_zeros'].sum()}")
    print()
    print(detail.to_string())
    print()
    return detail

diagnostic_zeros(df, "Hepatitis B")

Voici un code qui nous permet de détecter où se trouve le ratio idéal à appliquer à la colonne, pour la création d'histogramme par pays.    

In [ ]:
def ratio_mediane(df, colonne):
    """Ratio valeur / médiane du pays, pour chaque ligne."""
    # Médiane de chaque pays (les NaN sont ignorés par la médiane)
    mediane = df.groupby("Country")[colonne].transform("median")
    # Si la médiane vaut 0, la division donne inf : on remplace par NaN
    return (df[colonne] / mediane).replace([np.inf, -np.inf], np.nan)


def _exemples(df, colonne, ratio, masque, n):
    """Choisit n lignes réparties dans la tranche et les décrit avec leur contexte."""
    # On trie par pays/année pour que "précédente" et "suivante" aient un sens
    d = df.sort_values(["Country", "Year"]).copy()
    d["ratio"] = ratio.loc[d.index]
    d["mediane"] = d.groupby("Country")[colonne].transform("median")
    d["avant"] = d.groupby("Country")[colonne].shift(1)    # valeur de la ligne d'avant
    d["apres"] = d.groupby("Country")[colonne].shift(-1)   # valeur de la ligne d'après
 
    # Lignes de la tranche, triées par ratio croissant
    t = d[masque.loc[d.index]].sort_values("ratio")
 
    # n lignes régulièrement espacées (la plus basse, une du milieu, la plus haute...)
    positions = np.unique(np.linspace(0, len(t) - 1, min(n, len(t))).astype(int))
    texte = []
    for _, l in t.iloc[positions].iterrows():
        texte.append(f"{l['Country']} {int(l['Year'])} : {l['avant']:g} -> [{l[colonne]:g}] -> "
                     f"{l['apres']:g}  (médiane {l['mediane']:g})")
    return texte
 
 
def tableau_ratio(df, colonne, bornes=None, exemples=3):
    """Nombre de lignes par tranche de ratio, avec quelques exemples par tranche.
 
    Un exemple se lit : Pays Année : valeur d'avant -> [valeur de la ligne] -> valeur d'après (médiane du pays).
    Une valeur isolée entre deux valeurs proches = probable erreur ; une série qui monte
    ou descend d'un coup = probable vraie évolution.
    """
    ratio = ratio_mediane(df, colonne)
 
    if bornes is None:
        bornes = ([-0.001] + [round(0.05 * i, 2) for i in range(1, 21)]
                  + [1.25, 1.5, 2, 3, 5, float("inf")])
 
    tranche = pd.cut(ratio, bornes)
    comptes = tranche.value_counts().sort_index()
 
    for intervalle, n in comptes.items():
        print(f"{str(intervalle):<16}{n:>5} lignes")
        if n > 0 and exemples > 0:
            for ligne in _exemples(df, colonne, ratio, tranche == intervalle, exemples):
                print("      ", ligne)
    print("Ratio non calculable (NaN ou médiane = 0) :", ratio.isna().sum())
    
def histogramme_ratio(df, colonne, max_ratio=3, pas=0.05):
    """Histogramme du ratio valeur / médiane du pays.
    Barres de largeur `pas` ; les ratios > max_ratio sont regroupés dans la dernière barre."""
    ratio = ratio_mediane(df, colonne).dropna()
 
    # Plafonner à max_ratio : la dernière barre regroupe tous les ratios >= max_ratio
    ratio = ratio.clip(upper=max_ratio)
 
    fig = go.Figure(go.Histogram(
        x=ratio,
        xbins=dict(start=0, end=max_ratio + pas, size=pas),   # barres de largeur `pas`
    ))
    fig.update_layout(title=f"{colonne} : valeur / médiane du pays",
                      xaxis_title=f"Ratio (plafonné à {max_ratio})",
                      yaxis_title="Nombre de lignes")
    return fig


### **1) Traitement de la colonne Adult Mortality**

In [ ]:

diagnostic_zeros(df, "Adult Mortality")
tableau_ratio(df, "Adult Mortality")
histogramme_ratio(df, 'Adult Mortality')


Ici nous  avons un creux à 0.5, nous utiliserons donc un ratio de 0.5

In [ ]:
courbe_avec_outliers(df, 'Afghanistan', 'Adult Mortality', "Mort adulte 15 à 60 ans", 0.5)

### **2) Traitement de la colonne Alcohol**

In [ ]:
diagnostic_zeros(df, "Alcohol")
tableau_ratio(df, "Alcohol")

Nous constatons un creux vers 0.3, que nous utiliserons comme ratio.

In [ ]:
courbe_avec_outliers(df, 'France', 'Alcohol', "Consommation d'alcool par adultes en litres", 0.3)

### **3) Traitement de la colonne Hepatitis B**

In [ ]:
diagnostic_zeros(df, "Hepatitis B")
tableau_ratio(df, "Hepatitis B")

Nous utiliserons ici un ratio de 0,2.

In [ ]:
courbe_avec_outliers(df, 'Chad', 'Hepatitis B', "Taux de couverture vaccinale contre l'Hépatite B chez les enfants de 1 an (exprimé en %)", 0.2)

### **4) Traitement de la colonne Measles**

In [ ]:
diagnostic_zeros(df, "Measles")
tableau_ratio(df, "Measles")

Les données de la colonne Measles sont censé être un nombre de cas de rougeole pour 1000 personnes. Or beaucoup de mesures dépassent 1000. Après une vérification sur Internet, il est confirmé que ces chiffres sont un nombre de cas. Par ailleurs, la distribution des données ne permet pas de détecter un creux, il n'y aura pas d'utilisation d'un ration pour un graphique : nous n'allons donc appliquer aucun changement sur ces données, nous les gardons telle qu'elle.

### **5) Traitement de la colonne under_five deaths**

In [ ]:
diagnostic_zeros(df, "under-five deaths")
tableau_ratio(df, "under-five deaths")

Nous  choisissons de ne pas modifier les données d'"under-five deaths" car les valeurs sont trop petites pour rendre un ratio perinent. En effet les zéros sont en fait des valeurs de petits pays, l'unité est mal choisie pour mesurer cette donnée.     
Nous avons supprimé la colonne "infant deaths" car nous sommes partis du principe que ces valeurs étaient comprises dans la colonne "under-five deaths" : aucune valeurs de "infant deaths" ne dépasse "under-five deaths". Afin d'aller dans le sens de cette hypothèse, nous effectuons un test de corrélation entre ce deux colonnes. Par ailleurs ce code nous permet de calculer la part d'"infant deaths" dans "under-five deaths". La max de "1" montre que, sur une ligne, le maximum d'"infant deaths" est exactement "under-five deaths" : 

In [ ]:
def comparer_colonnes(df, a, b):
    """Compare deux colonnes : corrélation, et test « a est compris dans b » (a <= b)."""
    # On garde les lignes où les deux colonnes sont renseignées
    d = df[[a, b]].dropna()
    print(f"Lignes comparées : {len(d)}")

    # Corrélation : proche de 1 = les deux colonnes montent et descendent ensemble
    # Pearson : relation linéaire ; Spearman : ne regarde que l'ordre des valeurs
    print("Corrélation de Pearson  :", round(d[a].corr(d[b]), 4))
    print("Corrélation de Spearman :", round(d[a].corr(d[b], method="spearman"), 4))

    # Test d'inclusion : si a est compris dans b, a ne peut jamais dépasser b
    print(f"Lignes où {a} > {b} :", (d[a] > d[b]).sum())

    # Part de a dans b (seulement là où b est positif, sinon division par 0)
    positif = d[d[b] > 0]
    print(f"Part de {a} dans {b} :")
    print((positif[a] / positif[b]).describe().round(3).to_string())

comparer_colonnes(df,"infant deaths", "under-five deaths")

### **6) Traitement de la colonne Polio**

In [ ]:
diagnostic_zeros(df, "Polio")
tableau_ratio(df, "Polio")
histogramme_ratio(df, 'Polio', max_ratio=3, pas=0.05)

Nous utiliserons un ratio de 2,5 car il se toruve au milieu du creux. Les données avec un ratio proche de zéros représentent 171 lignes.      

### **7) Traitement de la colonne Total expenditure**

In [ ]:
diagnostic_zeros(df, "Polio")
tableau_ratio(df, "Polio")
histogramme_ratio(df, "Polio", max_ratio=3, pas=0.05)

### **8) Traitement de la colonne Dyphteria**

In [ ]:
diagnostic_zeros(df, "Diphtheria")
tableau_ratio(df, "Diphtheria")
histogramme_ratio(df, "Diphtheria", max_ratio=3, pas=0.05)

### **9) Traitement de la colonne GDP**

In [ ]:
diagnostic_zeros(df, "GDP")
tableau_ratio(df, "GDP")
histogramme_ratio(df, "GDP", max_ratio=3, pas=0.05)

In [ ]:
courbe_avec_outliers(df, 'Belarus', 'GDP', 'GDP Lituanie', 0.3)


### **10) Traitement de la Thiness 1-19 years**

Cette colonne reporte en fait le pourcentage d'enfants agés de 10 à 19 ans en état de maigreur.

In [ ]:
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)
diagnostic_zeros(df, "thinness 1-19 years")
tableau_ratio(df, "thinness 1-19 years")
histogramme_ratio(df, "thinness 1-19 years", max_ratio=3, pas=0.05)

### **11) Traitement de la Thiness 5-9 years**

In [ ]:
diagnostic_zeros(df, "thinness 5-9 years")
tableau_ratio(df, "thinness 5-9 years")
histogramme_ratio(df, "thinness 5-9 years", max_ratio=3, pas=0.05)

### **11) Traitement de la colonne Income composition of resources**

In [ ]:
diagnostic_zeros(df, "Income composition of resources")
tableau_ratio(df, "Income composition of resources")
histogramme_ratio(df, "Income composition of resources", max_ratio=3, pas=0.05)

### **13) Traitement de la colonne schooling**

In [ ]:
import plotly.express as px

# Une ligne par pays : on calcule la scolarité moyenne de chaque pays
pays = df.groupby("Country", as_index=False).agg(
    Schooling_moy=("Schooling", "mean")
)

# On retire les pays sans valeur, puis on trie pour que le graphe soit lisible
pays = pays.dropna().sort_values("Schooling_moy")

# Pays en abscisses, scolarité moyenne en ordonnées
fig = px.bar(pays, x="Country", y="Schooling_moy")
fig.show()

### **14) Traitement de la colonne life expectancy**

In [22]:
diagnostic_zeros(df, "Life expectancy")
tableau_ratio(df, "Life expectancy")
histogramme_ratio(df, "Life expectancy", max_ratio=3, pas=0.05)

--- Life expectancy ---
Zéros : 0 (0.0 %)
NaN   : 10
Aucun zéro à convertir.

(-0.001, 0.05]      0 lignes
(0.05, 0.1]         0 lignes
(0.1, 0.15]         0 lignes
(0.15, 0.2]         0 lignes
(0.2, 0.25]         0 lignes
(0.25, 0.3]         0 lignes
(0.3, 0.35]         0 lignes
(0.35, 0.4]         0 lignes
(0.4, 0.45]         0 lignes
(0.45, 0.5]         0 lignes
(0.5, 0.55]         0 lignes
(0.55, 0.6]         1 lignes
       Haiti 2010 : 62.5 -> [36.3] -> 62.3  (médiane 61.95)
(0.6, 0.65]         0 lignes
(0.65, 0.7]         0 lignes
(0.7, 0.75]         1 lignes
       Eritrea 2000 : nan -> [45.3] -> 58.1  (médiane 61.7)
(0.75, 0.8]         2 lignes
       Zambia 2000 : nan -> [43.8] -> 44.6  (médiane 56.55)
       Zambia 2001 : 43.8 -> [44.6] -> 45.5  (médiane 56.55)
(0.8, 0.85]        12 lignes
       Rwanda 2000 : nan -> [48.3] -> 48.6  (médiane 60.3)
       Botswana 2001 : 47.8 -> [46.7] -> 46  (médiane 57.2)
       Zambia 2004 : 46.4 -> [47.9] -> 49.3  (médiane 56.55)
(0.85, 0

### **15) Traitement de la colonne statut**

In [25]:
diagnostic_zeros(df, "Status")



--- Status ---
Zéros : 0 (0.0 %)
NaN   : 0
Aucun zéro à convertir.

